In [1]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import joblib


# ================================
# 2. Load Model and Data
# ================================
model = joblib.load("../model/churn_model.pkl")
X = pd.read_csv("../model/X_scaled.csv")
y = pd.read_csv("../model/y.csv")
y = y.values.ravel()

print("Model loaded:", type(model).__name__)
print("Data shape  :", X.shape)


# ================================
# 3. Create SHAP Explainer
# ================================
print("\nCreating SHAP explainer...")

explainer = shap.TreeExplainer(model)

# Sample 500 rows — standard practice for SHAP on large datasets
X_sample = shap.sample(X, 500, random_state=42)

print("Computing SHAP values on 500 samples...")
shap_values = explainer.shap_values(X_sample)

print("SHAP values computed successfully!")
print("SHAP values shape:", shap_values.shape)


# ================================
# 4. Summary Plot
# ================================
print("\nGenerating Summary Plot...")

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type="bar",
    show=False
)
plt.title("Feature Importance - SHAP Summary", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_summary_bar.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → shap_summary_bar.png")


# ================================
# 5. Beeswarm Plot
# ================================
print("\nGenerating Beeswarm Plot...")

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_sample,
    show=False
)
plt.title("SHAP Beeswarm - Feature Impact Direction", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → shap_beeswarm.png")


# ================================
# 6. High Risk Customer Explanation
# ================================
print("\nGenerating Individual Explanation...")

y_pred_proba = model.predict_proba(X_sample)[:, 1]
high_risk_idx = y_pred_proba.argmax()

print(f"Explaining customer index : {high_risk_idx}")
print(f"Churn probability         : {y_pred_proba[high_risk_idx]:.4f}")

shap.plots.waterfall(
    shap.Explanation(
        values        = shap_values[high_risk_idx],
        base_values   = explainer.expected_value,
        data          = X_sample.iloc[high_risk_idx],
        feature_names = X_sample.columns.tolist()
    ),
    show=False
)
plt.title(f"Why Customer {high_risk_idx} is High Risk", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_individual_highrisk.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → shap_individual_highrisk.png")


# ================================
# 7. Low Risk Customer Explanation
# ================================
low_risk_idx = y_pred_proba.argmin()

print(f"\nExplaining customer index : {low_risk_idx}")
print(f"Churn probability         : {y_pred_proba[low_risk_idx]:.4f}")

shap.plots.waterfall(
    shap.Explanation(
        values        = shap_values[low_risk_idx],
        base_values   = explainer.expected_value,
        data          = X_sample.iloc[low_risk_idx],
        feature_names = X_sample.columns.tolist()
    ),
    show=False
)
plt.title(f"Why Customer {low_risk_idx} is Low Risk", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig("../model/shap_individual_lowrisk.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved → shap_individual_lowrisk.png")


# ================================
# 8. Top SHAP Features
# ================================
print("\n--- Top 10 Features by SHAP Importance ---")

shap_importance = pd.DataFrame({
    'Feature'    : X_sample.columns,
    'SHAP_Value' : np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_Value', ascending=False)

print(shap_importance.head(10).to_string(index=False))

shap_importance.to_csv("../model/shap_feature_importance.csv", index=False)
print("\nSaved → shap_feature_importance.csv")

print("\nSHAP Analysis Complete!")
print("Files saved:")
print("  shap_summary_bar.png")
print("  shap_beeswarm.png")
print("  shap_individual_highrisk.png")
print("  shap_individual_lowrisk.png")
print("  shap_feature_importance.csv")

c:\Users\USER\Desktop\customer churn prediction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded: XGBClassifier
Data shape  : (7043, 30)

Creating SHAP explainer...
Computing SHAP values on 500 samples...
SHAP values computed successfully!
SHAP values shape: (500, 30)

Generating Summary Plot...
Saved → shap_summary_bar.png

Generating Beeswarm Plot...
Saved → shap_beeswarm.png

Generating Individual Explanation...
Explaining customer index : 375
Churn probability         : 0.9942
Saved → shap_individual_highrisk.png

Explaining customer index : 401
Churn probability         : 0.0001
Saved → shap_individual_lowrisk.png

--- Top 10 Features by SHAP Importance ---
                        Feature  SHAP_Value
                  Tenure Months    0.938484
   Internet Service_Fiber optic    0.698990
                Monthly Charges    0.679336
                 Dependents_Yes    0.558917
Payment Method_Electronic check    0.496247
              Contract_Two year    0.487448
             Multiple Lines_Yes    0.402283
               Streaming TV_Yes    0.327423
           Stream